# Is the arch stable? Rigid-block equilibrium

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps
%pip install -q compas_assembly==0.7.1

In [ ]:
import numpy as np
from scipy.optimize import linprog
from compas.geometry import Line, Point, dot_vectors, subtract_vectors
from compas_assembly.algorithms import assembly_interfaces
from compas_assembly.datastructures import Assembly
from compas_assembly.geometry import Arch
from compas_notebook import Viewer
from compas_notebook.config import Config

## Arch

In [ ]:
thickness = 0.3
assembly = Assembly.from_template(Arch(rise=3, span=10, thickness=thickness, depth=0.5, n=20))
assembly_interfaces(assembly, nmax=7, tmax=1e-3, amin=1e-2)
supports = sorted(assembly.nodes(), key=lambda node: assembly.node_point(node)[2])[:2]
free = [node for node in assembly.nodes() if node not in supports]
row = {node: 6 * i for i, node in enumerate(free)}

## Unknowns: normal and two tangential forces per interface vertex

In [ ]:
columns = []
for u, v in assembly.edges():
    for interface in assembly.edge_interfaces((u, v)):
        frame = interface.frame
        normal = frame.zaxis if dot_vectors(frame.zaxis, subtract_vectors(assembly.node_point(v), assembly.node_point(u))) > 0 else -frame.zaxis
        for point in interface.points:
            for direction in (normal, frame.xaxis, frame.yaxis):
                columns.append((u, v, point, direction))
len(columns)

## Equilibrium of each free block

In [ ]:
A = np.zeros((6 * len(free), len(columns)))
for j, (u, v, point, direction) in enumerate(columns):
    for node, sign in ((v, 1), (u, -1)):
        if node in row:
            arm = np.array(point) - np.array(assembly.node_point(node))
            A[row[node]:row[node] + 3, j] = sign * np.array(direction)
            A[row[node] + 3:row[node] + 6, j] = sign * np.cross(arm, direction)
b = np.zeros(6 * len(free))
for node in free:
    b[row[node] + 2] = assembly.node_block(node).volume()

## Friction and compression only

In [ ]:
mu = 0.6
normals = np.arange(0, len(columns), 3)
A_ub = []
for i in normals:
    for k in (1, 2):
        for sign in (1, -1):
            r = np.zeros(len(columns))
            r[i + k], r[i] = sign, -mu
            A_ub.append(r)
bounds = [(0, None) if j % 3 == 0 else (None, None) for j in range(len(columns))]
cost = np.zeros(len(columns))
cost[normals] = 1
result = linprog(cost, A_ub=np.array(A_ub), b_ub=np.zeros(len(A_ub)), A_eq=A, b_eq=b, bounds=bounds)
result.success

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [0, -14, 2]
config.view.camera.target = [0, 0, 1.5]
viewer = Viewer(config=config)
for node in assembly.nodes():
    viewer.scene.add(assembly.node_block(node), color="#cccccc", opacity=0.5)
if result.success:
    for j in normals:
        _, _, point, direction = columns[j]
        viewer.scene.add(Line(point, Point(*point) + direction * (-2 * result.x[j])), color="#dc4b8d")
viewer.show()